# ArcGIS + Python Snippets for Planners

Quick-reference Field Calculator expressions and standalone scripts for common GIS data tasks.

**How to use this notebook:**
- **Field Calculator expressions** — copy the code block into the Field Calculator. Multi-line `def` blocks go in the *Code Block* box; the final call (e.g. `get_corner(!YourFieldName!)`) goes in the *Expression* box below it. Select **Python** as the parser.
- **Standalone scripts** — run directly in this notebook or in any Python environment with `arcpy` available.

---

## String Operations

In [ ]:
# -------------------------------------------------------------------
# Extract text after a delimiter
# -------------------------------------------------------------------
# Splits a string on a marker (e.g. '--') and returns everything after it.
# Strips a trailing keyword (e.g. 'corner'). Null-safe.
#
# FIELD CALCULATOR usage:
#   Code Block: paste the def below
#   Expression: get_corner(!YourFieldName!)
# -------------------------------------------------------------------

def get_corner(value):
    if value is None:
        return None
    marker = "--"
    if marker not in value:
        return None
    part = value.split(marker, 1)[1].strip()
    part = part.replace("corner", "").strip()
    return part

# Example test (remove before using in Field Calculator)
print(get_corner("Main St -- NW corner"))  # → 'NW'
print(get_corner(None))                    # → None

In [ ]:
# -------------------------------------------------------------------
# Extract last N characters
# -------------------------------------------------------------------
# Returns the rightmost X characters — useful for isolating codes or suffixes.
#
# FIELD CALCULATOR expression (replace X with character count):
#   str(!YourFieldName!)[-X:]
#
# Example: last 4 characters
# -------------------------------------------------------------------

example = "BROOKLYN_BK"
print(example[-2:])   # → 'BK'
print(example[-4:])   # → '_BK' ... adjust X to your need

In [ ]:
# -------------------------------------------------------------------
# Concatenate two coordinate fields into a string
# -------------------------------------------------------------------
# FIELD CALCULATOR expression:
#   str(!LAT!) + ", " + str(!LON!)
# -------------------------------------------------------------------

lat = 40.7128
lon = -74.0060
print(str(lat) + ", " + str(lon))  # → '40.7128, -74.006'

---
## Numeric & Formatting

In [ ]:
# -------------------------------------------------------------------
# Zero-pad single-digit integers
# -------------------------------------------------------------------
# Formats integers below 10 with a leading zero.
# Useful for district codes, IDs, or ranked fields.
#
# FIELD CALCULATOR usage:
#   Code Block: paste the def below
#   Expression: format_district(!YourFieldName!)
#
# One-liner alternative:
#   str(!YourFieldName!).zfill(2)
# -------------------------------------------------------------------

def format_district(num):
    if num < 10:
        return f"0{num}"
    else:
        return str(num)

print(format_district(3))   # → '03'
print(format_district(12))  # → '12'

---
## Conditional Logic

In [ ]:
# -------------------------------------------------------------------
# Replace null with zero
# -------------------------------------------------------------------
# Returns the field value if populated, 0 if null.
# Prevents downstream math errors.
#
# FIELD CALCULATOR expression:
#   !A! if !A! is not None else 0
# -------------------------------------------------------------------

A = None
print(A if A is not None else 0)  # → 0

A = 42
print(A if A is not None else 0)  # → 42

In [ ]:
# -------------------------------------------------------------------
# Sum two fields, treating nulls as zero
# -------------------------------------------------------------------
# FIELD CALCULATOR expression:
#   (!A! if !A! is not None else 0) + (!B! if !B! is not None else 0)
#
# Extend for more fields:
#   ... + (!C! if !C! is not None else 0)
# -------------------------------------------------------------------

A, B = 10, None
result = (A if A is not None else 0) + (B if B is not None else 0)
print(result)  # → 10

In [ ]:
# -------------------------------------------------------------------
# Output 0 if any of N fields is zero
# -------------------------------------------------------------------
# Flags a record as ineligible (0) if any field is zero.
# Useful for composite suitability scoring.
#
# FIELD CALCULATOR expression:
#   0 if (!field1! == 0 or !field2! == 0 or !field3! == 0) else 1
# -------------------------------------------------------------------

field1, field2, field3 = 5, 0, 3
print(0 if (field1 == 0 or field2 == 0 or field3 == 0) else 1)  # → 0

In [ ]:
# -------------------------------------------------------------------
# Flag if any category field is non-zero
# -------------------------------------------------------------------
# Returns 1 if at least one of a set of category fields is populated.
#
# FIELD CALCULATOR expression:
#   1 if sum([
#       !cat1!, !cat2!, !cat3!, !cat4!, !cat5!,
#       !cat6!, !cat7!, !cat8!, !cat9!, !cat10!
#   ]) > 0 else 0
# -------------------------------------------------------------------

cat1, cat2, cat3 = 0, 0, 1
print(1 if sum([cat1, cat2, cat3]) > 0 else 0)  # → 1

In [ ]:
# -------------------------------------------------------------------
# Categorize numeric ranges into output values
# -------------------------------------------------------------------
# Assigns a score or code based on which range a value falls in.
#
# FIELD CALCULATOR usage:
#   Code Block: paste the def below
#   Expression: get_value(!YourFieldName!)
# -------------------------------------------------------------------

def get_value(x):
    if x < 100:
        return 0
    elif 100 <= x < 500:
        return 10
    elif 500 <= x < 1000:
        return 3
    else:  # x >= 1000
        return 1

for v in [50, 250, 750, 1200]:
    print(f"{v} → {get_value(v)}")

---
## Lookup & Value Mapping

In [ ]:
# -------------------------------------------------------------------
# NYC borough name → borough code
# -------------------------------------------------------------------
# Converts a borough name field to its standard numeric code (1–5).
# Returns None for unmatched values.
#
# FIELD CALCULATOR usage:
#   Code Block: paste the def below
#   Expression: get_borocode(!borough!)
# -------------------------------------------------------------------

def get_borocode(borough):
    codes = {
        "Manhattan": 1,
        "Bronx": 2,
        "Brooklyn": 3,
        "Queens": 4,
        "Staten Island": 5
    }
    return codes.get(borough, None)

print(get_borocode("Brooklyn"))      # → 3
print(get_borocode("Staten Island")) # → 5
print(get_borocode("Newark"))        # → None

In [ ]:
# -------------------------------------------------------------------
# Map categorical text to numeric values
# -------------------------------------------------------------------
# Converts a text category field to an ordered numeric value.
# Works for land use types, priority levels, road classifications, etc.
#
# FIELD CALCULATOR expression (no Code Block needed):
#   value_map = {"Value A": 1, "Value B": 2, ...}
#   value_map.get(!Category!, None)
# -------------------------------------------------------------------

value_map = {
    "Value A": 1,
    "Value B": 2,
    "Value C": 3,
    "Value D": 4,
    "Value E": 5
}

category = "Value C"
print(value_map.get(category, None))  # → 3

---
## Scoring & Normalization

In [ ]:
# -------------------------------------------------------------------
# Normalize a value to a 0–50 scale with floor and ceiling
# -------------------------------------------------------------------
# Linear normalization between a min threshold (100) and max threshold (800),
# clamped to a 0–50 output range.
#
# FIELD CALCULATOR usage:
#   Code Block: paste the def below
#   Expression: normalize(!YourFieldName!)
#
# Adjust 100 (min), 800 (max), and 50 (output scale) as needed.
# -------------------------------------------------------------------

def normalize(x):
    if x < 100:
        return 0
    elif x > 800:
        return 50
    else:
        return 50 * ((x - 100) / 700)

for v in [50, 100, 450, 800, 1000]:
    print(f"{v} → {normalize(v):.2f}")

In [ ]:
# -------------------------------------------------------------------
# Assign percentile tiers (1/2/3) across multiple score fields
# -------------------------------------------------------------------
# STANDALONE SCRIPT — run here, not in Field Calculator.
#
# For each score field, computes 33rd and 67th percentile cutoffs
# across all records, then writes a tier to a new paired _TIER field:
#   1 = low (below 33rd percentile)
#   2 = mid (33rd–67th percentile)
#   3 = high (above 67th percentile)
#
# Edit 'fc' path and 'value_fields' list before running.
# -------------------------------------------------------------------

import arcpy
import numpy as np

fc = r"path\to\your\featureclass"  # ← update this
value_fields = ["A_SCORE", "B_SCORE", "C_SCORE",
                "D_SCORE", "E_SCORE", "F_SCORE", "G_SCORE"]  # ← update this

# Create output tier fields if they don't already exist
for f in value_fields:
    out_field = f + "_TIER"
    if out_field not in [fld.name for fld in arcpy.ListFields(fc)]:
        arcpy.AddField_management(fc, out_field, "LONG")
        print(f"Added field: {out_field}")

# Compute percentile cutoffs for each field
cutoffs = {}
for field in value_fields:
    vals = [row[0] for row in arcpy.da.SearchCursor(fc, field)
            if row[0] is not None]
    cutoffs[field] = (np.percentile(vals, 33), np.percentile(vals, 67))
    print(f"{field}: p33={cutoffs[field][0]:.2f}, p67={cutoffs[field][1]:.2f}")

# Write tier values
update_fields = value_fields + [f + "_TIER" for f in value_fields]
with arcpy.da.UpdateCursor(fc, update_fields) as cursor:
    for row in cursor:
        for i, field in enumerate(value_fields):
            val = row[i]
            if val is None:
                row[i + len(value_fields)] = None
            else:
                p33, p67 = cutoffs[field]
                if val < p33:
                    row[i + len(value_fields)] = 1
                elif val < p67:
                    row[i + len(value_fields)] = 2
                else:
                    row[i + len(value_fields)] = 3
        cursor.updateRow(row)

print("Done — tiers computed for all fields.")

---
## Export

In [ ]:
# -------------------------------------------------------------------
# Export multiple layers to a single Excel workbook
# -------------------------------------------------------------------
# STANDALONE SCRIPT — run here, not in Field Calculator.
#
# Each layer becomes its own sheet. Sheet names are derived from
# the layer name (truncated to 31 chars per Excel's limit).
#
# Requires: pandas, openpyxl
#   pip install pandas openpyxl  (if not already in your environment)
#
# Edit 'out_xlsx' and 'layers' before running.
# -------------------------------------------------------------------

import arcpy
import pandas as pd
import os

out_xlsx = r"C:\path\to\output\combined_tables.xlsx"  # ← update this
layers = [
    r"C:\path\to\your.gdb\layer1",   # ← update these
    r"C:\path\to\your.gdb\layer2",
    r"C:\path\to\your.gdb\layer3",
]

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    for fc in layers:
        fields = [f.name for f in arcpy.ListFields(fc) if f.type != "Geometry"]
        arr = arcpy.da.TableToNumPyArray(fc, fields)
        df = pd.DataFrame(arr)
        sheet_name = os.path.basename(fc)[:31]  # Excel 31-char sheet name limit
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Exported: {sheet_name}")

print(f"\nDone → {out_xlsx}")

In [ ]:
# -------------------------------------------------------------------
# Export multiple layers to individual CSVs
# -------------------------------------------------------------------
# STANDALONE SCRIPT — run here, not in Field Calculator.
#
# Writes each layer to its own .csv file in a specified output folder.
# Uses native arcpy — no extra dependencies required.
# Geometry fields are excluded automatically.
#
# Edit 'out_folder' and 'layers' before running.
# -------------------------------------------------------------------

import arcpy
import os

out_folder = r"C:\path\to\output\csvs"  # ← update this
layers = [
    r"C:\path\to\your.gdb\layer1",   # ← update these
    r"C:\path\to\your.gdb\layer2",
    r"C:\path\to\your.gdb\layer3",
]

for fc in layers:
    name = os.path.basename(fc)
    print(f"Exporting {name}...")
    arcpy.conversion.TableToTable(
        in_rows=fc,
        out_path=out_folder,
        out_name=f"{name}.csv"
    )

print(f"\nDone → {out_folder}")